<a href="https://colab.research.google.com/github/sanmquin/AI/blob/main/src/Graphiko/Fetch-Business-Cluster-Videos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fetch videos for business-cluster channels
This notebook reuses the Graphiko business-cluster lookup logic and then fetches Finder videos grouped by channel.


In [ ]:
# Install pymongo if not already present
!pip install pymongo dnspython

import pymongo
from pymongo import MongoClient
from google.colab import userdata

try:
    # Retrieve the URI from Colab Secrets
    uri = userdata.get('MONGODB_URI')
    client = MongoClient(uri)

    # Send a ping to confirm connection
    client.admin.command('ping')
    print('✅ Successfully connected to MongoDB!')
except Exception as e:
    print(f'❌ MongoDB connection failed: {e}')


In [ ]:
# Access Finder DB collections and identify the latest business cluster
db = client['finder']
clusters_col = db['ChannelDescriptions_clusters']
items_col = db['ChannelDescriptions_items']
channels_col = db['channels']
videos_col = db['videos']

latest = clusters_col.find_one(sort=[('version', -1), ('createdAt', -1)])
latest_version = latest['version'] if latest else None
print('Latest cluster version:', latest_version)

if latest_version is None:
    print('No clusters found.')
    business_cluster = None
else:
    business_cluster = clusters_col.find_one({
        'version': latest_version,
        'name': {'$regex': '^business', '$options': 'i'}
    })

if business_cluster:
    business_cluster_id = business_cluster['_id']
    print('Business cluster found:', business_cluster.get('name'))
    print('Business cluster _id:', business_cluster_id)
else:
    business_cluster_id = None
    print('No business cluster found in the latest version.')


In [ ]:
# Resolve channels associated with the business cluster
if business_cluster_id is None:
    print('Cannot fetch channels without a business cluster id.')
    channel_ids = []
    business_channels = []
else:
    item_docs = list(items_col.find(
        {'clusterId': business_cluster_id},
        {'_id': 0, 'textId': 1}
    ))
    channel_ids = [d['textId'] for d in item_docs]

    business_channels = list(channels_col.find(
        {'channelId': {'$in': channel_ids}},
        {'_id': 0, 'channelId': 1, 'title': 1}
    ))

    print(f'Business cluster has {len(channel_ids)} channel ids and {len(business_channels)} channel docs.')
    for ch in sorted(business_channels, key=lambda x: x.get('title', ''))[:20]:
        print(f"- {ch.get('title', '(untitled)')} ({ch.get('channelId')})")

    if len(business_channels) > 20:
        print(f'... and {len(business_channels) - 20} more channels')


## Fetch videos for each business-cluster channel
Retrieves videos from Finder's `videos` collection for all channels in the business cluster and groups them by `channelId`.


In [ ]:
from collections import defaultdict

if not channel_ids:
    print('No channel ids available to query videos.')
    videos_by_channel = {}
else:
    cursor = videos_col.find(
        {'channelId': {'$in': channel_ids}},
        {
            '_id': 0,
            'videoId': 1,
            'channelId': 1,
            'title': 1,
            'publishedAt': 1,
            'viewCount': 1
        }
    ).sort([('channelId', 1), ('publishedAt', -1)])

    videos_by_channel = defaultdict(list)
    for doc in cursor:
        videos_by_channel[doc['channelId']].append(doc)

    print(f'Fetched videos for {len(videos_by_channel)} channels.')
    total_videos = sum(len(v) for v in videos_by_channel.values())
    print(f'Total videos fetched: {total_videos}')

    for channel in business_channels:
        cid = channel.get('channelId')
        title = channel.get('title', '(untitled)')
        channel_videos = videos_by_channel.get(cid, [])
        print(f'\n{title} ({cid}) -> {len(channel_videos)} videos')

        # Show up to the 3 most recent videos for quick inspection
        for video in channel_videos[:3]:
            print(f"  • {video.get('title', '(no title)')} | views={video.get('viewCount')} | publishedAt={video.get('publishedAt')}")


In [ ]:
# Optional: flatten all fetched videos to a DataFrame for downstream analysis/export
import pandas as pd

all_videos = [v for vids in videos_by_channel.values() for v in vids] if videos_by_channel else []
videos_df = pd.DataFrame(all_videos)
print('videos_df shape:', videos_df.shape)
videos_df.head(10)
